In [ ]:
groq_api_key = ""

### Free flow text output

In [4]:
from groq import Groq
import re

client = Groq(api_key=groq_api_key)


def register_employee(name, department, experience, skills):
    print("=== Employee Registered ===")
    print(name)
    print(department)
    print(experience)
    print(skills)


prompt = """
Extract employee information.

John Smith works in the AI department.
He has 8 years of experience.
His skills include Python, SQL and LangChain.
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

text = response.choices[0].message.content

print(text)

# ---------------------------------
# BAD PARSING
# ---------------------------------

name = re.search(r"Name:\s*(.*)", text)
department = re.search(r"Department:\s*(.*)", text)
experience = re.search(r"Experience:\s*(.*)", text)
skills = re.search(r"Skills:\s*(.*)", text)

register_employee(
    name.group(1),
    department.group(1),
    int(experience.group(1)),
    skills.group(1).split(",")
)

Here is the extracted employee information:

**Employee Name:** John Smith
**Department:** AI
**Years of Experience:** 8
**Skills:**

1. Python
2. SQL
3. LangChain


ValueError: invalid literal for int() with base 10: '** 8'

The AI answered correctly, but not in a format that my program could reliably consume. Prompt engineering isn't just about getting the right answer—it's about getting the answer in a predictable, machine-readable format.

### With Structured Output

In [9]:
from groq import Groq
import json

client = Groq(api_key=groq_api_key)


def register_employee(name, department, experience, skills):
    print("=== Employee Registered ===")
    print(name)
    print(department)
    print(experience)
    print(skills)


prompt = """
Extract employee information.

Return ONLY valid JSON.

Schema:

{
    "name":"",
    "department":"",
    "experience":0,
    "skills":[]
}

Text:

John Smith works in the AI department.
He has 8 years of experience.
His skills include Python, SQL and LangChain.
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    response_format={"type": "json_object"},
    messages=[
        {"role": "user", "content": prompt}
    ]
)

employee = json.loads(
    response.choices[0].message.content
)

print(employee)


{'name': 'John Smith', 'department': 'AI', 'experience': 8, 'skills': ['Python', 'SQL', 'LangChain']}


In [10]:
register_employee(
    employee["name"],
    employee["department"],
    employee["experience"],
    employee["skills"]
)

=== Employee Registered ===
John Smith
AI
8
['Python', 'SQL', 'LangChain']


### Notes

- Define the schema first (fields, types, required vs optional, allowed values).
- Prompt the model to fill the schema not to “write an answer.”
- Treat the output like an API response contract.